In [1]:
import mediapipe as mp
import numpy as np
import sys
import os
from decord import VideoReader, cpu
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import yaml
from collections import OrderedDict
from torchvision.transforms import v2
from lib.utils.model_utils import count_params, import_class
from lib.data.dataloader import CustomVideoDataset
from lib.utils.transforms import GetPoses_YOLO

c:\Users\ldezoetegrundy\AppData\Local\anaconda3\envs\vit_env\Lib\site-packages\torchvision\datapoints\__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
c:\Users\ldezoetegrundy\AppData\Local\anaconda3\envs\vit_env\Lib\site-packages\torchvision\transforms\v2\__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to

In [2]:
# Get arg file
with open('./config/custom_mppose/test_joint.yaml', 'r') as file:
    yaml_arg = yaml.safe_load(file)

class Dict2Class(object):
    def __init__(self, my_dict):
        for key in my_dict:
            setattr(self, key, my_dict[key])            

# Convert args
arg = Dict2Class(yaml_arg)

# Get the annotation file
with open(arg.test_feeder_args['label_path'], 'r') as file:
    ann_file = yaml.safe_load(file)

Create the dataset, later we'll create the dataloader

In [3]:
from ultralytics import YOLO
# Create pose detector
detector = YOLO('yolov8n-pose.pt')
transforms = v2.Compose([
    v2.Resize(size=(384,640)),
    v2.ToDtype(torch.float32),
    v2.ToTensor(),
    GetPoses_YOLO(detector=detector, max_frames=300, num_joints=17)
    ])
dataset = CustomVideoDataset(arg, transforms=transforms)

c:\Users\ldezoetegrundy\AppData\Local\anaconda3\envs\vit_env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:41: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `transforms.Compose([transforms.ToImageTensor(), transforms.ConvertImageDtype()])`.
  warnings.warn(


### Data input shape

data_path: the path to '.npy' data, the shape of data should be (N, C, T, V, M)

N (name/batch)

C (channels (x,y,score))

T (frame)

V (joint)

M (person)

Get the config arguments

In [5]:
from model import load_model
skel_model = load_model(arg)
skel_model.load_model()

Model total number of params: 2962071


## Inference!

In [9]:
import torch.optim as optim
from collections import defaultdict
from torch.utils.data import DataLoader

# Create the dataloader
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

# Get the parameters to optimise
param_groups = defaultdict(list)
for name, params in skel_model.model.named_parameters():
    param_groups['other'].append(params)
optim_param_groups = {
    'other': {'params': param_groups['other']}
}
params = list(optim_param_groups.values())

# Create the optimiser
optimizer = optim.SGD(
                params,
                lr=0.1,
                momentum=0.9,
                nesterov=False,
                weight_decay=0.0005)

loss = nn.CrossEntropyLoss()

In [7]:
classes = {}
for elem, key in enumerate(dict.fromkeys(key.split('_')[0] for key in ann_file.keys())):
    classes[key] = elem

Temporary dataloader (I pickled the tensors of the skeleton outputs)

***DELETE***

In [8]:
from torch.utils.data import Dataset
class QuickSet(Dataset):
    def __init__(self):
        self.root = './data/gestures/'
        with open(arg.test_feeder_args['label_path'], 'r') as file:
            self.ann_file = yaml.safe_load(file)
    
    def __len__(self):
        return len(self.ann_file.keys())

    def __getitem__(self, idx):
        return torch.load(f'{self.root}{list(self.ann_file.keys())[idx]}.pt'), self.ann_file[list(self.ann_file.keys())[idx]]

dataset = QuickSet()
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [10]:
from lib.training import train_simple_network
from sklearn.metrics import accuracy_score
torch.cuda.empty_cache()
device = torch.device('cuda:0')

score_funcs = {'Accuracy': accuracy_score}

train_simple_network(skel_model.model, loss, dataloader, score_funcs=score_funcs, epochs=2, device=device)

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/6 [00:00<?, ?it/s]

c:\Users\ldezoetegrundy\AppData\Local\anaconda3\envs\vit_env\Lib\site-packages\torchvision\transforms\functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(



WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 255.0. Dividing input by 255.
0: 384x640 1 person, 1: 384x640 1 person, 2: 384x640 1 person, 3: 384x640 1 person, 4: 384x640 1 person, 5: 384x640 1 person, 6: 384x640 1 person, 7: 384x640 1 person, 8: 384x640 1 person, 9: 384x640 1 person, 10: 384x640 1 person, 11: 384x640 1 person, 12: 384x640 1 person, 13: 384x640 1 person, 14: 384x640 1 person, 15: 384x640 1 person, 16: 384x640 1 person, 17: 384x640 1 person, 18: 384x640 1 person, 19: 384x640 1 person, 20: 384x640 1 person, 21: 384x640 1 person, 22: 384x640 1 person, 23: 384x640 1 person, 24: 384x640 1 person, 25: 384x640 1 person, 26: 384x640 1 person, 27: 384x640 1 person, 28: 384x640 1 person, 29: 384x640 1 person, 30: 384x640 1 person, 31: 384x640 1 person, 32: 384x640 1 person, 33: 384x640 1 person, 34: 384x640 1 person, 35: 384x640 1 person, 36: 384x640 1 person, 37: 384x640 1 person, 38: 384x640 1 person, 39: 384x640 1 person, 40: 384x640 1 person, 

Training:   0%|          | 0/6 [00:00<?, ?it/s]

c:\Users\ldezoetegrundy\AppData\Local\anaconda3\envs\vit_env\Lib\site-packages\torchvision\transforms\functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(



WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 255.0. Dividing input by 255.
0: 384x640 1 person, 1: 384x640 1 person, 2: 384x640 1 person, 3: 384x640 1 person, 4: 384x640 1 person, 5: 384x640 1 person, 6: 384x640 1 person, 7: 384x640 1 person, 8: 384x640 1 person, 9: 384x640 1 person, 10: 384x640 1 person, 11: 384x640 1 person, 12: 384x640 1 person, 13: 384x640 1 person, 14: 384x640 1 person, 15: 384x640 1 person, 16: 384x640 1 person, 17: 384x640 1 person, 18: 384x640 1 person, 19: 384x640 1 person, 20: 384x640 1 person, 21: 384x640 1 person, 22: 384x640 1 person, 23: 384x640 1 person, 24: 384x640 1 person, 25: 384x640 1 person, 26: 384x640 1 person, 27: 384x640 1 person, 28: 384x640 1 person, 29: 384x640 1 person, 30: 384x640 1 person, 31: 384x640 1 person, 32: 384x640 1 person, 33: 384x640 1 person, 34: 384x640 1 person, 35: 384x640 1 person, 36: 384x640 1 person, 37: 384x640 1 person, 38: 384x640 1 person, 39: 384x640 1 person, 40: 384x640 1 person, 

,epoch,total time,train loss,train Accuracy
0,0,67.043780,1.408044,0.333333
1,1,133.099493,1.224905,0.291667
